# 02 · Band-Split Experiments

Runs the Direction-03 experiment and reads its pre-registered verdict. Six new GPU
runs (mel ×3, uniform ×3 at the REDUCED budget) plus the **shared baseline cell**
(reused from Direction 01, *not* retrained), a per-band mechanism figure, a FULL
-budget budget-dependence guard, and one test pass. All training/eval cells are
**⚠️ RUN THIS LATER**; the analysis logic lives in `singnet/` and is unit-tested.
The notebook ships **un-executed**.

Companion to [`01_bandsplit_architecture.ipynb`](01_bandsplit_architecture.ipynb)
(the architecture + parameter match) and [`../THEORY.md`](../THEORY.md) §6 (the
statistics).

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (the pre-registered
  hypothesis — recapped verbatim below), §3.2 (run matrix + shared cell), §6
  (evaluation + the per-band mechanism analysis), §7 (run book + runtimes), §8
  (gates), §13 (interpretation matrix — the branch stubs).
- **Data prep is *not* repeated here** — Direction 01's notebooks own it; this
  direction reuses the 86/14/50 split verbatim.
- **Logic is in `singnet/`:** the ordered-chain verdict is
  `singnet/analysis/bandsplit.py`; the per-band metrics are
  `singnet/eval/banded.py`; the width/param table is `scripts/match_params.py`.
  Every one is exercised by the gate-G0 tests.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (verbatim from MASTER_PLAN §2)

> Let $\overline{v}(\text{arm})$ = mean best-checkpoint validation vocals SI-SDR
> over the arm's 3 seeds; σ_seed = pooled between-seed std over the three arms
> ($\sigma_{\text{seed}} = \sqrt{(s^2_{\text{base}} + s^2_{\text{uni}} + s^2_{\text{mel}})/3}$).
>
> **H-03 (ordered band-split benefit).** At matched parameters:
> - **Effect 1 (splitting helps):** $E_1 := \overline{v}(\text{uniform}) - \overline{v}(\text{baseline}) > \sigma_{\text{seed}}$.
> - **Effect 2 (mel spacing helps beyond splitting):** $E_2 := \overline{v}(\text{mel}) - \overline{v}(\text{uniform}) > \sigma_{\text{seed}}$.
> - **Fully supported** iff E1 **and** E2 (the ordered chain mel > uniform > baseline, each link outside the noise band).
> - **Partially supported** iff exactly one of E1/E2 holds (each partial case has a distinct pre-written interpretation, §13).
> - **Refuted (null)** iff neither holds with all three arms within ±σ_seed of each other — pre-registered headline: *"the band-split advantage is not detectable at ~10 M params / MUSDB-only / 16 k steps"*.
> - **Refuted (negative)** iff $\overline{v}(\text{baseline})$ exceeds both variants by > σ_seed — band isolation actively hurts at this scale (boundary-artifact reading, §13).
> Confirmation guard: the FULL-budget pair (§3.3) must not reverse the REDUCED-budget verdict's sign on its comparison; if it does, the verdict is downgraded to **mixed (budget-dependent)** — itself a pre-registered finding about short-training architecture comparisons.
>
> **Mechanism evidence (pre-registered, not a hypothesis):** the per-band error breakdown (§6.3). If H-03 holds, gains should concentrate in the vocal-energy-dense bands (~100 Hz–4 kHz); if gains are uniform across bands (or concentrated where vocals are absent), the dedicated-capacity mechanism is questioned even under a positive headline — we commit to saying so.
>
> **Descriptive secondaries (no hypotheses):** measured FLOPs/inference RTF per arm (at matched *parameters*, the band variants are analytically ~2× cheaper in encoder FLOPs); train-loss curves per arm (optimization-difficulty comparison).

## 2 · Run matrix & the shared-baseline accounting (§3.2)

| Stage | Runs | Budget | Seeds | New GPU runs |
|---|---|---|---|---|
| Sweep | `split_mel` ×3, `split_uniform` ×3 | REDUCED (16 k) | {0,1,2} | **6** |
| `baseline` | **0 new** — shared cell ≡ D01 `l1mag` sweep ≡ D02 `base` (config-hash equality asserted) | REDUCED | {0,1,2} | 0 |
| Confirmation | best variant + `baseline` | FULL (40 k) | {0} | **2** |
| Contingency | `baseline` + `split_mel` (only if D01 flips the default loss) | REDUCED | {0} | 0 or 2 |

Total new: **8** (+2 contingency). The baseline cell is reused, not retrained —
its config hash is bit-identical to D01's `l1mag_seed0_reduced` (asserted in
`tests/test_config_schema_d03.py`). Verify the shared cell before analysis:

In [ ]:
# CPU: confirm the shared-baseline hash equality (the reuse is provable, §3.2).
from singnet.utils.config import hash_config, resolve_config
d01 = hash_config(resolve_config("01-loss-function-study/configs/l1mag_seed0_reduced.yaml"))
d03 = hash_config(resolve_config("03-mini-band-split/configs/base.yaml"))
print("D01 l1mag_seed0_reduced :", d01)
print("D03 baseline cell       :", d03, " -> reused, not retrained" if d01 == d03 else " MISMATCH!")
assert d01 == d03

### 2.1 Pre-flight — dry-run the band edges, width, and parameter match

In [ ]:
# ⚠️ RUN THIS LATER (CPU, seconds) — prints band edges, chosen c, and the param
# count with its delta vs the 9,835,745 baseline for every split config.
# !python scripts/run_sweep.py --direction 03 --dry-run
# !python scripts/match_params.py --report        # regenerate results/param_match_table.md
print("Dry-run verifies the +-2% match and the partitions before any GPU spend (RUN LATER).")

### 2.2 Launch — smokes (G1) then the 6-run sweep

In [ ]:
# ⚠️ RUN THIS LATER (GPU, ~0.2 h each) — G1 smokes: both towers must overfit one
# chunk to > +20 dB train SI-SDR within 2k steps (same bar as the baseline).
# !python -m singnet.train --config 03-mini-band-split/configs/smoke_mel.yaml
# !python -m singnet.train --config 03-mini-band-split/configs/smoke_uniform.yaml
print("G1 smokes (RUN LATER). PASS = single-chunk overfit > +20 dB within 2k steps.")

In [ ]:
# ⚠️ RUN THIS LATER (GPU, ~1.3-1.8 h per run x6 = 8-11 h). Resumable; skips done runs.
# !python scripts/run_sweep.py --direction 03 --stage reduced
print("6 split runs (mel x3, uniform x3) at REDUCED budget (RUN LATER).")

## 3 · The ordered chain — H-03 verdict (registry-driven)

Once the 6 split runs and the 3 shared-baseline rows are in the registries, the
verdict is a pure function of the per-seed validation SI-SDR. `ordered_chain_verdict`
implements the §2 decision logic (unit-tested in `tests/test_bandsplit_stats.py`).

In [ ]:
import pandas as pd
from singnet.train import read_registry
from singnet.analysis import ordered_chain_verdict

# Baseline seeds come from Direction 01's l1mag sweep (the shared cell); the split
# arms from this direction's registry. (RUN LATER once both registries are filled.)
d03 = read_registry("03-mini-band-split/results/registry.csv")
d01 = read_registry("01-loss-function-study/results/registry.csv")

def seeds(frame, arm):
    rows = frame[(frame["arm"] == arm) & (frame["budget"] == 16000)]
    return rows.sort_values("seed")["best_val_sisdr"].astype(float).tolist()

baseline = seeds(d01, "l1mag")             # the shared baseline cell
uniform  = seeds(d03, "split_uniform")
mel      = seeds(d03, "split_mel")
if baseline and uniform and mel:
    v = ordered_chain_verdict(baseline, uniform, mel)
    print(f"means: baseline={v.mean_baseline:.3f}  uniform={v.mean_uniform:.3f}  mel={v.mean_mel:.3f}")
    print(f"sigma_seed={v.sigma_seed:.3f} dB")
    print(f"E1 (uniform-baseline)={v.e1:+.3f}  holds={v.e1_holds}")
    print(f"E2 (mel-uniform)     ={v.e2:+.3f}  holds={v.e2_holds}")
    print(f"VERDICT: {v.verdict}")
else:
    print("registries not yet populated — RUN LATER (this cell is inert until then).")

### 3.1 · Three-arm ranking with the σ_seed band

The house figure: mean validation SI-SDR per arm with the pooled σ_seed band
drawn, so a reader sees at a glance which links of the chain clear the noise.

In [ ]:
# RUN LATER — bar chart of the three arm means with the +-sigma_seed band and the
# E1/E2 links annotated. All numbers come from the verdict object above.
# import matplotlib.pyplot as plt
# arms = ["baseline", "uniform", "mel"]; means = [v.mean_baseline, v.mean_uniform, v.mean_mel]
# plt.bar(arms, means); plt.axhspan(... +-v.sigma_seed around each ...)
# plt.ylabel("val vocals SI-SDR (dB)"); plt.title(f"H-03: {v.verdict}")
print("3-arm ranking with the sigma_seed band renders once the registry is filled (RUN LATER).")

## 4 · Per-band mechanism figure (§6.3)

*Why* does (or doesn't) an arm win? For each system and track we score
band-limited SI-SDR and band magnitude error on a **fixed 6-band grid** (union of
both variants' interior edges + a 100 Hz floor split — layout-neutral), then plot
per-band **(variant − baseline)** deltas. Pre-registered reading (§2): under a
positive headline, gains should concentrate in the vocal-dense bands
(~100 Hz–4 kHz); gains in the wrong bands question the dedicated-capacity story
even if the headline is positive.

In [ ]:
# ⚠️ RUN THIS LATER (CPU minutes, needs the FULL checkpoints + decoded shards).
# from singnet.eval import banded_report, analysis_grid_hz
# grid = analysis_grid_hz()                    # the fixed 6-band grid (Hz)
# for each validation/test track: separate with baseline & best variant, then
#   rep = banded_report(vocals_ref, est_variant) ; rep_b = banded_report(vocals_ref, est_baseline)
#   delta = rep["band_sisdr"] - rep_b["band_sisdr"]     # per-band (variant - baseline)
# Aggregate deltas across tracks (mean + per-track spread) and plot vs band centre.
print(f"6-band mechanism grid (Hz): computed by singnet.eval.analysis_grid_hz (RUN LATER).")

## 5 · Budget-dependence guard (REDUCED vs FULL)

The primary verdict is on REDUCED (16 k). The FULL (40 k) pair — best variant +
baseline, seed 0 — must **not reverse the sign** of the headline comparison; if it
does, the verdict downgrades to **mixed (budget-dependent)** (§2, a finding in its
own right).

In [ ]:
# ⚠️ RUN THIS LATER (GPU, ~3.5-4.5 h x2). Generates confirm_*_full.yaml from the
# frozen reduced ranking (best variant + the reused baseline) and runs them.
# !python scripts/run_sweep.py --direction 03 --stage full
# Then check: sign(FULL: best_variant - baseline) == sign(REDUCED: same). Else -> mixed.
print("FULL-budget confirmation pair (RUN LATER); guards the 'needed longer' objection.")

## 6 · Efficiency table (descriptive secondary, §6.4)

Params (from the committed match table), analytic MACs (THEORY §4.5), and — RUN
LATER — measured MACs (a counting hook on one 6-s chunk) and CPU inference RTF for
one validation track. Reported as a **bonus**, never the claim.

In [ ]:
# CPU-runnable: params (exact) + the analytic ~0.54x encoder-MAC ratio (THEORY §4.5).
from singnet.models import SingNetC1, BandSplitUNet
base_p = SingNetC1().num_parameters
var_p = BandSplitUNet.from_mel_bands().num_parameters
print(f"params: baseline={base_p:,}  variants={var_p:,}  (+{100*(var_p-base_p)/base_p:.3f}%)")
print("analytic encoder MACs ratio (variant/baseline) ~= 0.54  -> ~1.85x cheaper (THEORY §4.5)")
# ⚠️ RUN THIS LATER (CPU): measured MACs via a forward hook on one 6-s chunk, and
# wall-clock inference RTF for one validation track — the descriptive secondary.

## 7 · The single test pass (§6.2)

Exactly **one** test-set pass for this direction: the 2 FULL-budget checkpoints
(best variant + baseline) on the 50 test tracks, per-track CSV committed, with a
paired-by-track bootstrap 95 % CI + Wilcoxon on the one pre-registered pair. Sweep
arms never touch test (grep-audited, gate G3).

In [ ]:
# ⚠️ RUN THIS LATER (GPU minutes / CPU) — the ONE test pass for this direction.
# !python scripts/evaluate.py --checkpoint <best_variant.pt> --split test --banded
# !python scripts/evaluate.py --checkpoint <baseline.pt>     --split test --banded
# Paired per-track delta (variant - baseline): bootstrap 95% CI + Wilcoxon signed-rank.
print("Single test pass + banded breakdown + paired stats (RUN LATER).")

## 8 · Interpretation — pre-written branches (select one when results exist)

These are **pre-registered** (MASTER_PLAN §13); the branch that matches the
verdict is adopted, the others struck. Full prose lives in `paper/PAPER.md`.

### If **fully supported** (mel > uniform > baseline, both links > σ_seed)
The partition idea itself transfers to compact scale, and *where* capacity goes
matters — the SOTA family's core idea is scale-robust. **Consequence:** adopt the
mel-split front-end as an option for the shipped model; the ≈ ½ encoder-FLOPs bonus
strengthens the deployment story. Confirm the per-band gains sit in the vocal-dense
bands (§4) before claiming the mechanism.

### If **partial — E1 only** (splitting helps, mel ≈ uniform)
Dedicated capacity is the active ingredient; perceptual spacing is not (at 3
bands). **Consequence:** prefer the simpler uniform split; flag 3-band granularity
as the suspect for mel's null (a band-count follow-up, not run here).

### If **partial — E2 only** (mel > uniform, but uniform ≤ baseline)
Splitting alone does nothing/hurts; mel spacing *rescues* it — capacity
*concentration*, not isolation, is the mechanism. **Consequence:** intriguing; the
per-band figure adjudicates; flag for a band-count/overlap follow-up.

### If **refuted (null)** — all three within ±σ_seed
**Headline:** the band-split advantage is a large-model / large-data phenomenon —
not detectable at ~10 M params on MUSDB at this budget. **Consequence:** the
baseline stays; the project's capacity-vs-data narrative (with D02's verdict)
sharpens; an honest contribution to the field's replication discourse (cite the
BSRNN replication study, arXiv 2603.09187).

### If **refuted (negative)** — baseline beats both by > σ_seed
Band isolation costs more than dedicated capacity buys at this scale (the
boundary-artifact reading; the per-band figure should show damage at
band-boundary-adjacent bands, THEORY §3). **Consequence:** baseline stays; document
the failure mode the SOTA family engineered around (band overlaps, cross-band MLPs).

### If **budget-dependent** — FULL reverses REDUCED
Short-budget architecture comparisons mislead — a methodological finding.
**Consequence:** report both budgets; recommend future architecture comparisons at
FULL only.

### If **mechanism mismatch** — positive headline, gains in the wrong bands
The effect is real but not the dedicated-capacity story. **Consequence:** say so;
propose the band-count/overlap follow-up without running it.

## 9 · Conclusions — what the verdict changes for the project

Direction 03 asks whether the SOTA family's **core idea** (band-split, dedicated
per-band capacity) survives when everything else is held fixed and the size is
matched — the controlled comparison the literature never runs. Whatever the sign:

- A **positive** result adds a cheap (≈ ½ encoder-FLOPs) front-end option and shows
  the idea is scale-robust.
- A **null** is a genuine, pre-registered finding — "band-split is a large-model
  phenomenon" — that, together with Direction 02's data-scaling verdict, sharpens
  the project's **capacity-vs-data** narrative: at ~10 M params on MUSDB-only, what
  moves vocals SI-SDR is (D02) data and (D03) *not* the band partition per se.
- A **negative** documents the failure mode (boundary artifacts) the SOTA family
  engineered around, motivating overlaps / cross-band coupling for a follow-up.

The contribution is the **controlled, param-matched, seeded** comparison itself —
the honest framing the field's own replication study (arXiv 2603.09187) argues for
— not the direction of the effect.

Next in the project: the verdict feeds `paper/PAPER.md` (stage E) and the
capacity-vs-data synthesis with Direction 02.